In [22]:
!pip install ultralytics

In [23]:
import cv2
import numpy as np
import torch
import random
from collections import Counter
import os
import shutil
import yaml

from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.utils.loss import v8DetectionLoss

In [ ]:
INPUT_PATH = '/kaggle/input/datasets/hazeldiv/icdas-caries-2'
OUTPUT_PATH = '/kaggle/working/preprocessed'
MODEL_YAML_PATH = '/kaggle/working/yolo26n-p5-disabled.yaml'

TRAIN_IMG_DIR = f'{INPUT_PATH}/train/images'
TRAIN_LBL_DIR = f'{INPUT_PATH}/train/labels'
VAL_IMG_DIR = f'{INPUT_PATH}/valid/images'
VAL_LBL_DIR = f'{INPUT_PATH}/valid/labels'
TEST_IMG_DIR = f'{INPUT_PATH}/test/images'
TEST_LBL_DIR = f'{INPUT_PATH}/test/labels'

In [25]:
class CLAHETransform:
    def __init__(self, clip_limit=4.0, tile_grid_size=(8, 8)):
        self.clip_limit = clip_limit
        self.tile_grid_size = tile_grid_size
        self.clahe = cv2.createCLAHE(clipLimit=self.clip_limit, tileGridSize=self.tile_grid_size)

    def __call__(self, img):
        if img is None:
            return img
        lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
        lab[:, :, 0] = self.clahe.apply(lab[:, :, 0])
        return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

In [26]:
class ImageAugmentor:
    def __init__(self):
        pass
    
    def horizontal_flip(self, img):
        return cv2.flip(img, 1)
    
    def rotate(self, img, angle):
        h, w = img.shape[:2]
        center = (w // 2, h // 2)
        matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
        return cv2.warpAffine(img, matrix, (w, h), borderMode=cv2.BORDER_REFLECT)
    
    def brightness_contrast(self, img, alpha=1.0, beta=0):
        return cv2.convertScaleAbs(img, alpha=alpha, beta=beta)
    
    def random_augment(self, img):
        aug_img = img.copy()
        if random.random() > 0.5:
            aug_img = self.horizontal_flip(aug_img)
        angle = random.uniform(-10, 10)
        aug_img = self.rotate(aug_img, angle)
            
        alpha = random.uniform(0.9, 1.1)
        beta = random.uniform(-10, 10)
        aug_img = self.brightness_contrast(aug_img, alpha, beta)
            
        return aug_img

In [27]:
os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(f'{OUTPUT_PATH}/train/images', exist_ok=True)
os.makedirs(f'{OUTPUT_PATH}/train/labels', exist_ok=True)
os.makedirs(f'{OUTPUT_PATH}/valid/images', exist_ok=True)
os.makedirs(f'{OUTPUT_PATH}/valid/labels', exist_ok=True)
os.makedirs(f'{OUTPUT_PATH}/test/images', exist_ok=True)
os.makedirs(f'{OUTPUT_PATH}/test/labels', exist_ok=True)

In [28]:
clahe = CLAHETransform(clip_limit=4.0, tile_grid_size=(8, 8))

train_count = 0
for img_file in os.listdir(TRAIN_IMG_DIR):
    if img_file.startswith('.'):
        continue
    img_path = f'{TRAIN_IMG_DIR}/{img_file}'
    img = cv2.imread(img_path)
    if img is not None:
        img_clahe = clahe(img)
        cv2.imwrite(f'{OUTPUT_PATH}/train/images/{img_file}', img)
        train_count += 1
    base_name = os.path.splitext(img_file)[0]
    lbl_file = f'{base_name}.txt'
    src_lbl = f'{TRAIN_LBL_DIR}/{lbl_file}'
    if os.path.exists(src_lbl):
        shutil.copy(src_lbl, f'{OUTPUT_PATH}/train/labels/{lbl_file}')

print(f"Processed {train_count} training images with CLAHE")

val_count = 0
for img_file in os.listdir(VAL_IMG_DIR):
    if img_file.startswith('.'):
        continue
    img_path = f'{VAL_IMG_DIR}/{img_file}'
    img = cv2.imread(img_path)
    if img is not None:
        img_clahe = clahe(img)
        cv2.imwrite(f'{OUTPUT_PATH}/valid/images/{img_file}', img)
        val_count += 1
    base_name = os.path.splitext(img_file)[0]
    lbl_file = f'{base_name}.txt'
    src_lbl = f'{VAL_LBL_DIR}/{lbl_file}'
    if os.path.exists(src_lbl):
        shutil.copy(src_lbl, f'{OUTPUT_PATH}/valid/labels/{lbl_file}')

print(f"Processed {val_count} validation images with CLAHE")

test_count = 0
for img_file in os.listdir(TEST_IMG_DIR):
    if img_file.startswith('.'):
        continue
    img_path = f'{TEST_IMG_DIR}/{img_file}'
    img = cv2.imread(img_path)
    if img is not None:
        img_clahe = clahe(img)
        cv2.imwrite(f'{OUTPUT_PATH}/test/images/{img_file}', img)
        test_count += 1
    base_name = os.path.splitext(img_file)[0]
    lbl_file = f'{base_name}.txt'
    src_lbl = f'{TEST_LBL_DIR}/{lbl_file}'
    if os.path.exists(src_lbl):
        shutil.copy(src_lbl, f'{OUTPUT_PATH}/test/labels/{lbl_file}')

print(f"Processed {test_count} test images with CLAHE")

Processed 697 training images with CLAHE
Processed 197 validation images with CLAHE
Processed 106 test images with CLAHE


In [29]:
original_counts = Counter()
images_by_class = {i: [] for i in range(7)}

for lbl_file in os.listdir(f'{OUTPUT_PATH}/train/labels'):
    if not lbl_file.endswith('.txt'):
        continue
    base_name = os.path.splitext(lbl_file)[0]
    classes_in_file = set()
    with open(f'{OUTPUT_PATH}/train/labels/{lbl_file}') as f:
        for line in f:
            if line.strip():
                cls = int(line.strip().split()[0])
                original_counts[cls] += 1
                classes_in_file.add(cls)
    for cls in classes_in_file:
        images_by_class[cls].append(base_name)

class_names = ['D0-Sound', 'D1-Faint-Enamel', 'D2-Distinct-Enamel', 
                'D3-Enamel-Breakdown', 'D4-Dentinal-Shadow', 
                'D5-Distinct-Cavity', 'D6-Extensive-Cavity']
print("Original class distribution:")
for cls, count in sorted(original_counts.items()):
    print(f"Class {cls} ({class_names[cls]}): {count} instances")

min_count = min(original_counts.values())
print(f"\nMinority class count: {min_count}")

Original class distribution:
Class 0 (D0-Sound): 4930 instances
Class 1 (D1-Faint-Enamel): 1443 instances
Class 2 (D2-Distinct-Enamel): 727 instances
Class 3 (D3-Enamel-Breakdown): 454 instances
Class 4 (D4-Dentinal-Shadow): 236 instances
Class 5 (D5-Distinct-Cavity): 231 instances
Class 6 (D6-Extensive-Cavity): 231 instances

Minority class count: 231


In [30]:
augmentor = ImageAugmentor()
augmented_count = 0
augmented_per_class = Counter()

OVERSAMPLE_RATIO = 2
TARGET_PER_CLASS = {cls: int(min_count * OVERSAMPLE_RATIO) for cls in original_counts.keys()}

augmentations_needed = {}
for cls in original_counts.keys():
    current = original_counts[cls]
    target = TARGET_PER_CLASS[cls]
    augmentations_needed[cls] = max(0, target - current)

print("Augmentations needed per class:")
for cls, n in sorted(augmentations_needed.items()):
    print(f"Class {cls} ({class_names[cls]}): {n}")

images_already_augmented = set()

for cls, n_augments in sorted(augmentations_needed.items(), key=lambda x: -x[1]):
    if n_augments <= 0:
        continue
    
    available_images = [img for img in images_by_class[cls] if img not in images_already_augmented]
    
    if not available_images:
        continue
    
    augs_per_image = max(1, n_augments // len(available_images))
    
    for img_base in available_images:
        if augmented_per_class[cls] >= n_augments:
            break
        
        img_files = [f for f in os.listdir(f'{OUTPUT_PATH}/train/images') 
                     if os.path.splitext(f)[0] == img_base]
        if not img_files:
            continue
        
        ext = os.path.splitext(img_files[0])[1]
        img = cv2.imread(f'{OUTPUT_PATH}/train/images/{img_files[0]}')
        if img is None:
            continue
        
        for i in range(augs_per_image):
            if augmented_per_class[cls] >= n_augments:
                break
            
            aug_img = augmentor.random_augment(img)
            aug_name = f'{img_base}_cls{cls}_aug_{i}{ext}'
            cv2.imwrite(f'{OUTPUT_PATH}/train/images/{aug_name}', aug_img)
            
            lbl_src = f'{OUTPUT_PATH}/train/labels/{img_base}.txt'
            lbl_dst = f'{OUTPUT_PATH}/train/labels/{img_base}_cls{cls}_aug_{i}.txt'
            shutil.copy(lbl_src, lbl_dst)
            
            augmented_count += 1
            for c in original_counts.keys():
                with open(lbl_src) as f:
                    for line in f:
                        if line.strip():
                            if int(line.strip().split()[0]) == c:
                                augmented_per_class[c] += 1
            
            images_already_augmented.add(img_base)

print(f"\nGenerated {augmented_count} augmented images")

final_counts = Counter()
for lbl_file in os.listdir(f'{OUTPUT_PATH}/train/labels'):
    if not lbl_file.endswith('.txt'):
        continue
    with open(f'{OUTPUT_PATH}/train/labels/{lbl_file}') as f:
        for line in f:
            if line.strip():
                final_counts[int(line.strip().split()[0])] += 1

print("\nFinal class distribution:")
for cls, count in sorted(final_counts.items()):
    print(f"Class {cls} ({class_names[cls]}): {count} instances")

Augmentations needed per class:
Class 0 (D0-Sound): 0
Class 1 (D1-Faint-Enamel): 0
Class 2 (D2-Distinct-Enamel): 0
Class 3 (D3-Enamel-Breakdown): 8
Class 4 (D4-Dentinal-Shadow): 226
Class 5 (D5-Distinct-Cavity): 231
Class 6 (D6-Extensive-Cavity): 231

Generated 312 augmented images

Final class distribution:
Class 0 (D0-Sound): 5941 instances
Class 1 (D1-Faint-Enamel): 1827 instances
Class 2 (D2-Distinct-Enamel): 987 instances
Class 3 (D3-Enamel-Breakdown): 609 instances
Class 4 (D4-Dentinal-Shadow): 428 instances
Class 5 (D5-Distinct-Cavity): 421 instances
Class 6 (D6-Extensive-Cavity): 453 instances


In [31]:
data_config = {
    'path': OUTPUT_PATH,
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': 7,
    'names': class_names
}

yaml_path = f'{OUTPUT_PATH}/data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_config, f, sort_keys=False)

print(f"Created YAML config at {yaml_path}")

Created YAML config at /kaggle/working/preprocessed/data.yaml


In [32]:
class WiseIoU:
    def __init__(self, eps=1e-7):
        self.eps = eps

    def __call__(self, pred, target):
        px1, py1, px2, py2 = pred.chunk(4, -1)
        tx1, ty1, tx2, ty2 = target.chunk(4, -1)
        
        inter_w = (torch.min(px2, tx2) - torch.max(px1, tx1)).clamp(0)
        inter_h = (torch.min(py2, ty2) - torch.max(py1, ty1)).clamp(0)
        inter = inter_w * inter_h
        union = (px2 - px1) * (py2 - py1) + (tx2 - tx1) * (ty2 - ty1) - inter + self.eps
        iou = inter / union
        
        cx_pred = (px1 + px2) / 2
        cy_pred = (py1 + py2) / 2
        cx_true = (tx1 + tx2) / 2
        cy_true = (ty1 + ty2) / 2
        rho2 = (cx_pred - cx_true) ** 2 + (cy_pred - cy_true) ** 2
        
        cw = torch.max(px2, tx2) - torch.min(px1, tx1)
        ch = torch.max(py2, ty2) - torch.min(py1, ty1)
        
        dist_penalty = rho2 / (cw ** 2 + ch ** 2 + self.eps)
        
        r = torch.exp(dist_penalty).clamp(max=10.0)
        
        wise_iou = iou * r
        
        return wise_iou.clamp(min=0, max=3.0)

In [33]:
class CustomWIoULoss(v8DetectionLoss):
    def __init__(self, model):
        super().__init__(model)
        self.wise_iou = WiseIoU()

    def bbox_loss(self, pred_dist, pred_bboxes, anchor_points, target_bboxes, 
                  target_scores, target_scores_sum, fg_mask):
        if fg_mask.sum() == 0:
            return torch.tensor(0.0, device=pred_bboxes.device), \
                   torch.tensor(0.0, device=pred_bboxes.device)
        loss_iou = (self.wise_iou(pred_bboxes[fg_mask], target_bboxes[fg_mask]) 
                    * target_scores.sum(-1)[fg_mask].unsqueeze(-1)).sum() / target_scores_sum
        loss_dfl = self.dfl(pred_dist[fg_mask], target_bboxes[fg_mask], 
                           anchor_points, target_scores, target_scores_sum, fg_mask)
        return loss_iou, loss_dfl

In [34]:
class CustomTrainer(DetectionTrainer):
    def get_criterion(self):
        return CustomWIoULoss(self.model)

In [ ]:
yolo26n_p5_disabled = {
    "nc": 80,
    "end2end": True,
    "reg_max": 1,
    "scales": {"n": [0.50, 0.25, 1024]},
    "backbone": [
        [-1, 1, "Conv", [64, 3, 2]],
        [-1, 1, "Conv", [128, 3, 2]],
        [-1, 2, "C3k2", [256, False, 0.25]],
        [-1, 1, "Conv", [256, 3, 2]],
        [-1, 2, "C3k2", [512, False, 0.25]],
        [-1, 1, "Conv", [512, 3, 2]],
        [-1, 2, "C3k2", [512, True]],
        [-1, 1, "SPPF", [512, 5]],
        [-1, 2, "C2PSA", [512]],
    ],
    "head": [
        [-1, 1, "nn.Upsample", [None, 2, "nearest"]],
        [[-1, 4], 1, "Concat", [1]],
        [-1, 2, "C3k2", [256, False]],
        [[11, 8], 1, "Detect", ["nc"]],
    ],
}

with open(MODEL_YAML_PATH, "w") as file:
    yaml.dump(yolo26n_p5_disabled, file, sort_keys=False, default_flow_style=None)

In [ ]:
model = YOLO(MODEL_YAML_PATH)

results = model.train(
    data=yaml_path,
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    optimizer='SGD',
    project='runs/train',
    name='caries_yolo26n_v3',
    exist_ok=True,
    patience=50,
    save=True,
    save_period=10,
    val=True,
    plots=True
)

Ultralytics 8.4.70 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/preprocessed/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=caries_yolo26n_v3, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=Tr

In [36]:
import glob

best_models = glob.glob(f'/kaggle/working/runs/detect/runs/train/caries_yolo26n_v3/weights/best.pt')
if best_models:
    best_model_path = best_models[0]
    model = YOLO(best_model_path)
    test_results = model.val(data=yaml_path, split='test')
    
    print('\n=== Test Set Results ===')
    print(f"mAP50: {test_results.box.map50:.4f}")
    print(f"mAP50-95: {test_results.box.map:.4f}")
    print(f"Precision: {test_results.box.mp:.4f}")
    print(f"Recall: {test_results.box.mr:.4f}")
    
    print('\nPer-class AP (mAP50):')
    for i, ap in enumerate(test_results.box.ap50):
        print(f"  {class_names[i]}: {ap:.4f}")
else:
    print("Best model not found. Check training output.")

Ultralytics 8.4.70 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO26n summary (fused): 122 layers, 2,376,201 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1907.4±455.2 MB/s, size: 112.3 KB)
val: Scanning /kaggle/working/preprocessed/test/labels... 148 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 148/148 1.2Kit/s 0.1s<0.1s
val: New cache created: /kaggle/working/preprocessed/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 2.9it/s 3.5s.2ss
                   all        148       1373       0.56       0.47      0.473      0.256
              D0-Sound        126        812      0.712      0.842      0.819      0.497
       D1-Faint-Enamel        103        276      0.469      0.449      0.408      0.221
    D2-Distinct-Enamel         60        108      0.535      0.278      0.295      0.154
   D3-Enamel-Breakdown         46         66      